# ELM corpus: charge exchange to the isobaric analog state

$(p,n)$ scattering to the isobaric analog state of the target ground state. The analog
state is the isospin partner of the ground state, so this transition constrains the
isovector part of the optical potential in a way elastic scattering alone does not.

Selecting it is the whole difficulty. The analog state is picked out by a window on the
residual excitation energy, but EXFOR entries report that energy inconsistently -- some
give an excitation energy, some a Q value, some a level or analog-state number, and some
nothing at all. Entries that fall outside the window for those reasons are re-added
below, one at a time, on explicit grounds.

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

import numpy as np
from periodictable import elements

from nn_corpora import elm, elm_curate, munge, plotting, serialize, spec

## Targets

The near-spherical targets of the elastic sectors, further restricted to those with a
tabulated analog-state excitation energy.

In [ ]:
targets = spec.elm_pn_targets()
for A, Z in sorted(targets, key=lambda t: (t[1], t[0])):
    print(f"  {A}{elements[Z].symbol:<3s}  E_IAS = {spec.ELM_EX_IAS[(A, Z)]:8.4f} MeV")

## Query

The excitation-energy window is $\pm$ 0.3 MeV about the tabulated analog state. Unlike
the elastic sectors there is no minimum point count: analog-state angular distributions
are sparse, sometimes only a few angles.

In [ ]:
pn = elm_curate.query_pn(
    targets,
    einc_range=spec.ELM_PN_EINC_RANGE,
    ias_window=spec.ELM_IAS_WINDOW,
)
for target, multi in sorted(pn.items()):
    entries = multi.data["dXS/dA"]
    print(f"{target}: {len(entries.entries)} parsed, {len(entries.failed_parses)} failed")

## Repair failed parses

In [ ]:
result = elm_curate.ElmSectorResult(sector="charge_exchange")
elm_curate.repair_failed_parses(pn, ("dXS/dA",), result)
print("\n".join(sorted(set(result.repaired))) or "nothing to repair")

## Entries the excitation-energy window rejects

`O0178` reports the analog state by analog-state number rather than excitation energy,
so it never falls inside any window; it is re-added with no excitation-energy
restriction. `D0049` (Carlson) is older and reports the analog state energy differently
from modern compilations, so its window is widened to $\pm$ 0.5 MeV. `O0090` (Batty)
reports lab-frame angles, converted to the CM frame below. `O0138` reports asymmetric
$\pm$`ERR-T` uncertainties.

In [ ]:
for target in [(48, 20), (208, 82), (120, 50), (90, 40)]:
    if target in pn:
        elm_curate.readd_entry(
            pn, target, "O0178", result,
            einc_range=spec.ELM_PN_EINC_RANGE,
            parsing_kwargs=elm.PARSE_RECIPES["O0178"].parsing_kwargs,
            reason="reports the analog state by analog-state number, not excitation energy",
        )

for target in [(208, 82), (118, 50), (120, 50), (90, 40)]:
    if target in pn:
        ex = spec.ELM_EX_IAS[target]
        elm_curate.readd_entry(
            pn, target, "D0049", result,
            einc_range=spec.ELM_PN_EINC_RANGE,
            ex_range=(ex - elm.WIDE_IAS_WINDOW, ex + elm.WIDE_IAS_WINDOW),
            reason=elm.WIDE_IAS_WINDOW_ENTRIES["D0049"],
        )

for target in [(116, 50), (124, 50)]:
    if target in pn:
        elm_curate.readd_entry(
            pn, target, "O0138", result,
            einc_range=spec.ELM_PN_EINC_RANGE,
            parsing_kwargs=elm.PARSE_RECIPES["O0138"].parsing_kwargs,
            reason="reports asymmetric +/-ERR-T uncertainties",
        )

print("\n".join(result.repaired))

### Lab-frame angles

`O0090` (Batty et al.) reports lab-frame angles. It is re-added with the frame filter
off and converted here; the conversion is exact for two-body kinematics with the
projectile lighter than the target.

In [ ]:
for target in [(208, 82), (120, 50)]:
    if target not in pn:
        continue
    elm_curate.readd_entry(
        pn, target, "O0090", result,
        einc_range=spec.ELM_PN_EINC_RANGE,
        parsing_kwargs=elm.PARSE_RECIPES["O0090"].parsing_kwargs,
        reason=elm.LAB_ANGLE_ENTRIES["O0090"],
    )
    entry = pn[target].data["dXS/dA"].entries.get("O0090")
    if entry is not None:
        elm_curate.convert_lab_angles(entry, target, spec.PROJECTILES["proton"])
        print(f"{target}: converted O0090 to CM angles")

## Exclusions and duplicate measurements

`T0162` is dominated by Gamow-Teller strength rather than the analog-state transition.
`E1667` carries a duplicate empty measurement ahead of the real one.

In [ ]:
elm_curate.exclude_entries(pn, "dXS/dA", elm.EXCLUDED_PN, result)

for entry_id, index, reason in elm.MEASUREMENT_DROPS:
    for target, multi in pn.items():
        entry = multi.data["dXS/dA"].entries.get(entry_id)
        if entry is not None and len(entry.measurements) > index + 1:
            del entry.measurements[index]
            result.excluded.append(f"{entry_id} measurement {index}: {reason}")

print("\n".join(sorted(set(result.excluded))) or "no exclusions applied")

## Missing uncertainties

`C0134` reports no uncertainty for several points. Figure 5 of Phys. Rev. C 31 (1985)
1147 shows the markers there are larger than the error bars, and the last point without
an error has a value close to the first point with one. The missing uncertainties are
therefore set, conservatively, to about the size of the first reported one.

In [ ]:
elm_curate.apply_uncertainty_patches(pn, result)
print("\n".join(sorted(set(result.repaired))))

## Munge and serialize

In [ ]:
elm_curate.finalize(pn, "charge_exchange", "proton", result)
print(result.summary())

## Inspect

In [ ]:
for target, multi in sorted(pn.items()):
    measurements = [m for e in multi.data["dXS/dA"].entries.values() for m in e.measurements]
    if measurements:
        plotting.plot_angular(
            measurements, n_per_plot=6,
            title=plotting._latex(f"{target[0]}{elements[target[1]].symbol}") + " (p,n) IAS")
plt.show()

## Write

In [ ]:
serialize.write_sector(result.records, corpus="elm", sector="charge_exchange")
print(f"wrote {len(result.records)} measurements to data/elm/charge_exchange/")